In [ ]:
# import pandas --- spreadsheet library
import pandas as pd
import numpy as np
import os

# import matplotlib --- plotting library
import matplotlib.pyplot as plt

from plotting import make_overview, compare, smoothen
from myio import read_in_weather_data

In [ ]:
import sys
sys.path.append('../PalmSim')
sys.path.append('..')
from palmsim import PalmField

In [ ]:
%matplotlib inline

In [ ]:
def read_in_observed_yield(party, fn):
    """ Read in a csv of observed yields and return the df. """
    
    fdir = '../../Data/Calibration Data/{:}/Clean data'.format(party)

    ydir = 'Yield'

    fp = os.path.join(fdir,ydir,fn)

    # observed yield
    dfo = pd.read_csv(fp)
    dfo = dfo.set_index(pd.to_datetime(dfo['Month']))
    dfo = dfo.dropna(how='all')
    dfo['Y (kg/ha/yr)'] = 1000*dfo['Y (t/ha/mo)']*(365/dfo.index.days_in_month)
    
    return dfo

# a small test to check the function
assert read_in_observed_yield('Sinarmas', 'Hnae-1986.csv') is not None

In [ ]:
def read_in_weather(party, fnw):
    """ Read in a csv of observed weather and return the df. """
    
    fdir = '../../Data/Calibration Data/{:}/Clean data'.format(party)

    # weather data
    wdir = 'Weather'
    fpw = os.path.join(fdir,wdir,fnw)

    dfw = read_in_weather_data(fpw)
    dfw['T-avg'] = 0.5*(dfw['T-max'] + dfw['T-min'])
    
    return dfw

# a small test to check the function
assert read_in_weather('Sinarmas', 'HNAE.csv') is not None

In [ ]:
def list_files(party, kind='Yield'):
    """ Return a list of filenames in folder of 'party' and 'kind'. 
    
    >>> list_files('Sinarmas', 'Yield')
    ['Hnae-1986.csv',
    ...,
    ...]
    """
    fdir = '../../Data/Calibration Data/{:}/Clean data/{:}'.format(party, kind)
    return os.listdir(fdir)

assert list_files('Sinarmas') is not None

In [ ]:
def run_model(dfw,
              YOP=1999,
              MOP=1,
              density=140,
              WHC = 400,
              latitude=0,
              water_limited=True):
    """ Run the model given weather data and other inputs.
    
    Returns a dataframe containing run results.
    """

    # 1. Initialize the model
    dt = 10

    p = PalmField(year_of_planting=YOP, month_of_planting=MOP, planting_density = density, dt=dt)

    p.soil.parameters['water_holding_capacity']['value'] = WHC

    # 2. Couple the solar radiation data:
    dfw = dfw.resample('D').pad()
    dfw = dfw.rolling(dt, center=True).mean()
    dfw = dfw.fillna(method='pad')

    p.weather.radiation_series = dfw['Solar']
    
    if water_limited:
        p.weather.rainfall_series = dfw['Precip']
    else:
        p.weather.rainfall = 12
        
    p.weather.humidity_series = dfw['RelHum']
    p.weather.temperature_series = dfw['T-avg']
    p.weather.windspeed_series = dfw['wind']
    
    nyears = 30

    duration = nyears*365

    df = p.run(duration=duration)
    
    return df

In [ ]:
# our overview table of observed yield metadata
# we iterate over this file to process all model runs
overview = pd.read_excel('summary_table.xls')

In [ ]:
overview = overview.T.to_dict()

In [ ]:
from palmsim.components.soil import PTF

In [ ]:
res = []

N = len(overview)
i = 1

# number of files processed, used to show run progress
ndone = 0

# set to false to not do any runs, for debugging
run = True

for index, data in overview.items():
    
    # the 'info' variable is used to log model inputs
    info = data
        
    party = data['Source']
    YOP = data['Year of planting']
    
    MOP = data['Month of planting']
    
    # month of planting - default := 1
    if np.isnan(MOP):
        MOP = 1
    else:
        MOP = int(MOP)
    
    fn = data['Filename_clean']
    fnw = data['Filename_weatherdata_clean']
    
    density = data['Density']
    
    latitude = data['Lattitude (degrees)']
    
    use = data['Use']
    
    # some output
    s = ' '.join([str(x) for x in [fn, fnw, density]])    
    print('({:}/{:}: {:})'.format(i,N,s))
    
    depth = data['Soil depth']
    
    # in meters
    if isinstance(depth, str):
        
        # >100cm -> 1 (meter)
        depth = float(depth.replace('>','').replace('cm', '').strip())
        depth *= 0.01 # cm -> m
    else:
        print('\t Cant use - no soil depth')
        use = False
    
    texture = data['Soil texture']
    
    # soil water holding capacity in rootable zone (mm)
    if isinstance(texture, str):
        
        texture = texture.lower().strip()        
        ptf = PTF(texture_class=texture)
    
        # in mm/m
        PAWC = 1000*ptf.plant_available_water_content
        
        # in mm
        WHC = depth*PAWC        
                
    else:
        print('\t Cant use - no soil depth')
        use = False
        
    island = data['Island']
       
    name = fnw.split('.')[0]   
    
    info['WHC'] = WHC
    
    if run and use:
        
        try:
            
            dfo = read_in_observed_yield(party, fn)
            dfw = read_in_weather(party, fnw)

            yr0 = dfo.index[0].year - YOP
            yr1 = dfo.index[-1].year - YOP

            info['YAP-obs-start'] = yr0
            info['YAP-obs-end'] = yr1  

            info['Q.9-FFB'] = round(dfo['Y (kg/ha/yr)'].quantile(.9),-3)

            columns_to_keep = ['generative_bunch_count (1/ha/mo)',
                   'generative_bunch_weight (kg)',
                   'generative_FFB_production (t/ha/yr)']
            
            # calculate Yw
            
            dfyw = run_model(dfw,
                           YOP=YOP,
                           MOP=MOP,
                           density=density,
                           WHC = WHC,
                           latitude=latitude,
                           water_limited=True
                          )
            
            dfyw[columns_to_keep].to_csv(os.path.join("results","water-limited","{:}".format(fn)))
            
            # calculate Yp
            
            dfyp = run_model(dfw,
                           YOP=YOP,
                           MOP=MOP,
                           density=density,
                           WHC = WHC,
                           latitude=latitude,
                           water_limited=False
                          )
            
            dfyp[columns_to_keep].to_csv(os.path.join("results","potential","{:}".format(fn)))

            # also store Ya
            dfo.to_csv(os.path.join("results","observed","{:}".format(fn)))
            
            ndone += 1
            
        except FileNotFoundError:
            print('\t File not found...')
            pass
        
        except Exception:
            print('\t Warning: Something went wrong...')
            pass
    
    i += 1
    
    break;
    

In [ ]:
overview = pd.DataFrame(res)

In [ ]:
overview.to_csv('overview_results.csv')

In [ ]:
dfw